# 02 - Preprocessing

Notebook nay dung de lam sach du lieu cho bai toan sentiment analysis.

Quyet dinh:

- Chi dung 2 cot `Review` va `Rating`
- Chi giu `Rating` la so nguyen tu 1 den 5
- Tao nhan `sentiment` tu `Rating`
- Luu ket qua vao `data/processed/clean_reviews.csv`

## Buoc 1 - Import thu vien

In [ ]:
import re
import pandas as pd
from pathlib import Path

## Buoc 2 - Khai bao duong dan

In [ ]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "restaurant_reviews.csv"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "clean_reviews.csv"

print("Raw data path:", RAW_DATA_PATH)
print("Processed data path:", PROCESSED_DATA_PATH)
print("Raw file exists:", RAW_DATA_PATH.exists())

## Buoc 3 - Doc du lieu raw

In [ ]:
df = pd.read_csv(RAW_DATA_PATH)

print("So dong ban dau:", df.shape[0])
print("So cot ban dau:", df.shape[1])

## Buoc 4 - Chi lay 2 cot can dung

`Review` la input cho model. `Rating` dung de tao nhan sentiment.

In [ ]:
df_clean = df[["Review", "Rating"]].copy()

df_clean.head()

## Buoc 5 - Xoa dong thieu Review hoac Rating

Dong thieu `Review` khong co input. Dong thieu `Rating` khong tao duoc label.

In [ ]:
rows_before = len(df_clean)

missing_review = df_clean["Review"].isna()
missing_rating = df_clean["Rating"].isna()


df_clean = df_clean.dropna(subset=["Review", "Rating"]).copy()

rows_after = len(df_clean)
print("Thieu Review:", missing_review.sum())
print("Thieu Rating:", missing_rating.sum())
print("Thieu Review hoac Rating:", (missing_review | missing_rating).sum())
print("Thieu ca Review va Rating:", (missing_review & missing_rating).sum())
print("So dong truoc khi xoa missing:", rows_before)
print("So dong sau khi xoa missing:", rows_after)

## Buoc 6 - Chuyen Rating sang dang so

`errors="coerce"` bien gia tri khong chuyen duoc sang so thanh `NaN`, vi du `Like`.

In [ ]:
df_clean["Rating"] = pd.to_numeric(df_clean["Rating"], errors="coerce")

df_clean["Rating"].value_counts(dropna=False).sort_index()

## Buoc 7 - Chi giu Rating nguyen tu 1 den 5

Chi dung cac rating ro rang: `1`, `2`, `3`, `4`, `5`.

Cac gia tri nhu `1.5`, `2.5`, `3.5`, `4.5`, `Like` se bi loai bo.

In [ ]:
rows_before = len(df_clean)

valid_ratings = [1, 2, 3, 4, 5]
df_clean = df_clean[df_clean["Rating"].isin(valid_ratings)].copy()
df_clean["Rating"] = df_clean["Rating"].astype(int)

rows_after = len(df_clean)
print("So dong truoc khi loc rating:", rows_before)
print("So dong sau khi loc rating:", rows_after)
print("So dong da xoa:", rows_before - rows_after)

df_clean["Rating"].value_counts().sort_index()

## Buoc 8 - Xoa review trung lap

Neu cung mot noi dung review lap lai nhieu lan, ta chi giu lan dau tien.

In [ ]:
rows_before = len(df_clean)

duplicates = df_clean["Review"].duplicated().sum()
print("So dong trung lap:", duplicates)


df_clean = df_clean.drop_duplicates(subset=["Review"]).copy()

rows_after = len(df_clean)
print("So dong truoc khi xoa duplicate:", rows_before)
print("So dong sau khi xoa duplicate:", rows_after)
print("So dong da xoa:", rows_before - rows_after)


## Buoc 9 - Lam sach text co ban

Tao cot `review_clean` de dung cho model sau nay.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


df_clean["review_clean"] = df_clean["Review"].apply(clean_text)

df_clean[["Review", "review_clean"]].head()

## Buoc 10 - Tao nhan sentiment tu Rating

Quy tac:

- `1`, `2` -> `Negative`
- `3` -> `Neutral`
- `4`, `5` -> `Positive`

In [ ]:
def rating_to_sentiment(rating):
    if rating in [1, 2]:
        return "Negative"
    if rating == 3:
        return "Neutral"
    return "Positive"


df_clean["sentiment"] = df_clean["Rating"].apply(rating_to_sentiment)

df_clean["sentiment"].value_counts()

## Buoc 11 - Kiem tra du lieu sau preprocessing

In [ ]:
df_clean.head()

In [ ]:
df_clean.info()

## Buoc 12 - Luu du lieu da lam sach

File output se duoc dung cho notebook EDA va train model.

In [ ]:
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(PROCESSED_DATA_PATH, index=False)

print("Da luu file:", PROCESSED_DATA_PATH)
print("So dong:", df_clean.shape[0])
print("So cot:", df_clean.shape[1])